# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets by their @id and fields

record_sets = dataset.record_sets()
if not record_sets:
    print("No record sets found in this dataset. Please check the Croissant schema for record set definitions.")
else:
    for rs in record_sets:
        print(f"Record set name: {rs.name}, @id: {rs.id}")
        if hasattr(rs, 'fields') and rs.fields:
            print("  Fields:")
            for fld in rs.fields:
                print(f"    Field name: {fld.name}, @id: {fld.id}, dataType: {getattr(fld, 'data_type', None)}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Identify record sets by @id
record_sets = dataset.record_sets()
dataframes = {}
loaded_at_least_one = False

for rs in record_sets:
    try:
        rs_id = rs.id
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            loaded_at_least_one = True
            print(f"\nRecord set '@id': {rs_id}")
            print(f"Columns ({len(df.columns)}): {df.columns.tolist()}")
            print(df.head())
        else:
            print(f"\nRecord set '@id': {rs_id} contains no records.")
    except Exception as e:
        print(f"Could not load record set {rs.id}: {e}")

if not loaded_at_least_one:
    print("No records loaded for any record set. Please consult the dataset documentation and ensure records exist.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# EDA: Choose a record set and numeric field for demonstration.

# Please update `chosen_record_set_id` and `numeric_field_id` after reviewing section 2 and 3 for your actual record set and field @ids
if dataframes:
    chosen_record_set_id = list(dataframes.keys())[0]  # Pick the first available record set
    df = dataframes[chosen_record_set_id]
    numeric_field_id = None
    for col in df.columns:
        # Heuristic: select the first float/int-like column
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    if numeric_field_id:
        print(f"Using record set '@id': {chosen_record_set_id} and numeric field '{numeric_field_id}'\n")

        threshold = df[numeric_field_id].mean() # Example threshold, can be replaced
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize
        normalized = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = normalized
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical column if present
        group_field = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean {numeric_field_id} by '{group_field}':")
            print(grouped_df.head())
    else:
        print(f"No numeric field found in DataFrame for record set '@id': {chosen_record_set_id}.")
else:
    print("No dataframes to analyze. Please run the previous cell and ensure records are loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualization example: Histogram and scatter plot
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'chosen_record_set_id' in locals() and numeric_field_id:
    df = dataframes[chosen_record_set_id]

    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, color="dodgerblue", edgecolor="black")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Scatter plot example if at least one other numeric field
    other_numeric = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) and col != numeric_field_id]
    if other_numeric:
        plt.figure(figsize=(6, 6))
        sns.scatterplot(x=df[numeric_field_id], y=df[other_numeric[0]], alpha=0.6)
        plt.title(f"{numeric_field_id} vs {other_numeric[0]}")
        plt.xlabel(numeric_field_id)
        plt.ylabel(other_numeric[0])
        plt.show()
else:
    print("No dataframe or suitable fields for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to use the `mlcroissant` library to load a Croissant-formatted dataset, review its structure, load records using the `@id` of record sets, and conduct basic exploratory data analysis and visualization on the dataset. For more detailed analysis, refer to the [mlcroissant documentation](https://mlcommons.github.io/croissant/).